## LexisNexis Fraud Detection - Task 3: Random Forest

Trains a random forest using the imbalance strategy chosen in Task 2 (class weighting) and evaluates it on the 1995 hold-out (validation) set with the same metrics as the logistic regression baseline. Setup (data, features, preprocessing) is copied from Task 1. All preprocessing is fit on the 1994 training set only. The 1996 test set is not touched here.

In [1]:
!rm -rf LexisNexis-1A-detecting-fraudulent-insurance-claims
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [2]:
import json
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# M2 Task 1 saves both of these so every notebook uses identical data and
# identical features. Do not copy feature code between notebooks: it already
# drifted, Task 2's copy was missing NewPolicy and this one was not.
df = pd.read_csv("data/fraud_oracle_engineered.csv")

with open("data/model_features.json") as f:
    MODEL_FEATURES = json.load(f)

TARGET = "FraudFound_P"
print("Loaded", len(MODEL_FEATURES), "features from data/model_features.json")

train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
# 1996 is deliberately not loaded. It gets opened once, in Task 6.

X_train, y_train = train[MODEL_FEATURES], train[TARGET]
X_val,   y_val   = val[MODEL_FEATURES],   val[TARGET]

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

# only group columns that are in the feature list. PolicyType was dropped at
# the Challenge Advisor's request, so this guard stops the ColumnTransformer
# looking for a column that is not there.
grouped_categories = [c for c in ["Make", "PolicyType"]
                      if c in categorical_features]
regular_categories = [c for c in categorical_features
                      if c not in grouped_categories]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                             min_frequency=0.01)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, regular_categories),
    ("grouped_categorical", grouped_category_pipeline, grouped_categories),
])

preprocessor.fit(X_train)          # training year only
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
feature_names = preprocessor.get_feature_names_out()

print("Grouped:", grouped_categories)
print("Train:", X_train_processed.shape, " Validation:", X_val_processed.shape)

Loaded 24 features from data/model_features.json
Grouped: ['Make']
Train: (6141, 88)  Validation: (5195, 88)


Task 3: Random forest with class weighting

Same `evaluate` function as Task 2, so every row uses identical metrics: precision, recall and F1 at a 0.5 threshold, PR-AUC, and the share of fraud captured in the top 10% / 20% highest-scored claims.

The logistic regression rows are re-run here on the same 129 features (Task 1 includes `NewPolicy`, which the Task 2 notebook left out), so the comparison is like-for-like.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score)

CAPACITY = 0.20   # how much of the claim book a review team can get through


def evaluate(name, model, X_tr, y_tr, X_ev, y_ev):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]

    pred_half = (proba >= 0.5).astype(int)
    thresh = np.quantile(proba, 1 - CAPACITY)
    pred_cap = (proba >= thresh).astype(int)

    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    cap10 = y_sorted[:int(len(y_sorted) * 0.10)].sum() / y_sorted.sum()
    cap20 = y_sorted[:int(len(y_sorted) * 0.20)].sum() / y_sorted.sum()

    return {
        "Approach": name,
        "PR-AUC": average_precision_score(y_ev, proba),
        "Caught top 10%": cap10,
        "Caught top 20%": cap20,
        "Precision @cap": precision_score(y_ev, pred_cap, zero_division=0),
        "Recall @cap": recall_score(y_ev, pred_cap),
        "F1 @cap": f1_score(y_ev, pred_cap),
        "Precision @0.5": precision_score(y_ev, pred_half, zero_division=0),
        "Recall @0.5": recall_score(y_ev, pred_half),
        "F1 @0.5": f1_score(y_ev, pred_half),
    }


results = []

# logistic regression reference rows, carried over from Task 2
results.append(evaluate("LR, no handling",
                        LogisticRegression(max_iter=1000, random_state=42),
                        X_train_processed, y_train, X_val_processed, y_val))
results.append(evaluate("LR, class_weight balanced",
                        LogisticRegression(class_weight="balanced",
                                           max_iter=1000, random_state=42),
                        X_train_processed, y_train, X_val_processed, y_val))

**Random forest settings.** `class_weight="balanced"` applies the Task 2 strategy. `min_samples_leaf=20` stops trees from growing down to single-claim leaves: fully grown trees memorize the 409 training frauds and almost never output a probability above 0.5 on new claims, which makes recall collapse. 500 trees keeps the scores stable.

In [4]:
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
results.append(evaluate("RF, class_weight balanced", rf,
                        X_train_processed, y_train, X_val_processed, y_val))

# Task 2 found that imbalance handling does not improve ranking for logistic
# regression. Trees handle it differently, so test it here rather than
# assuming. balanced_subsample rebalances inside each tree instead of once.
results.append(evaluate("RF, no class weight",
                        RandomForestClassifier(n_estimators=500,
                                               min_samples_leaf=20,
                                               random_state=42, n_jobs=-1),
                        X_train_processed, y_train, X_val_processed, y_val))
results.append(evaluate("RF, balanced_subsample",
                        RandomForestClassifier(n_estimators=500,
                                               min_samples_leaf=20,
                                               class_weight="balanced_subsample",
                                               random_state=42, n_jobs=-1),
                        X_train_processed, y_train, X_val_processed, y_val))

# fully grown trees, kept as a reference for the 0.5 threshold point
results.append(evaluate("RF, fully grown trees",
                        RandomForestClassifier(n_estimators=500,
                                               class_weight="balanced",
                                               random_state=42, n_jobs=-1),
                        X_train_processed, y_train, X_val_processed, y_val))

In [5]:
results_df = pd.DataFrame(results).set_index("Approach").round(4)

print("RANKING QUALITY. Lead with this.\n")
display(results_df[["PR-AUC", "Caught top 10%", "Caught top 20%"]])

print(f"\n\nAT A CUTOFF THAT FLAGS THE TOP {int(CAPACITY*100)}% OF CLAIMS.")
print("This is how the model would actually be used.\n")
display(results_df[["Precision @cap", "Recall @cap", "F1 @cap"]])

print("\n\nAT THE DEFAULT 0.5 CUTOFF. Shown only to demonstrate why it misleads.")
print("Fully grown trees rarely score above 0.5, so recall collapses even")
print("though their ranking is fine.\n")
display(results_df[["Precision @0.5", "Recall @0.5", "F1 @0.5"]])

RANKING QUALITY. Lead with this.



,PR-AUC,Caught top 10%,Caught top 20%
Approach,,,
"LR, no handling",0.1488,0.2824,0.5216
"LR, class_weight balanced",0.1393,0.2724,0.5083
"RF, class_weight balanced",0.1448,0.2857,0.5183
"RF, no class weight",0.1431,0.2724,0.5216
"RF, balanced_subsample",0.1453,0.2857,0.5116
"RF, fully grown trees",0.1556,0.3223,0.5415




AT A CUTOFF THAT FLAGS THE TOP 20% OF CLAIMS.
This is how the model would actually be used.



,Precision @cap,Recall @cap,F1 @cap
Approach,,,
"LR, no handling",0.1511,0.5216,0.2343
"LR, class_weight balanced",0.1473,0.5083,0.2284
"RF, class_weight balanced",0.1501,0.5183,0.2328
"RF, no class weight",0.1511,0.5216,0.2343
"RF, balanced_subsample",0.1482,0.5116,0.2299
"RF, fully grown trees",0.1536,0.5415,0.2394




AT THE DEFAULT 0.5 CUTOFF. Shown only to demonstrate why it misleads.
Fully grown trees rarely score above 0.5, so recall collapses even
though their ranking is fine.



,Precision @0.5,Recall @0.5,F1 @0.5
Approach,,,
"LR, no handling",0.2500,0.0033,0.0066
"LR, class_weight balanced",0.1265,0.7176,0.2150
"RF, class_weight balanced",0.1407,0.6213,0.2294
"RF, no class weight",0.0000,0.0000,0.0000
"RF, balanced_subsample",0.1430,0.6279,0.2329
"RF, fully grown trees",0.2500,0.0033,0.0066


### Improving the Random Forest Model: Hyperparameter Tuning

We will use `GridSearchCV` to tune key hyperparameters of the Random Forest. We'll optimize for `average_precision` (PR-AUC) since it is a robust metric for highly imbalanced fraud detection tasks.

In [6]:
from sklearn.model_selection import GridSearchCV

# min_samples_leaf now starts at 1, which is the fully grown forest. The
# original grid started at 10, so it could never find the setting that
# actually scores best on 1995.
param_grid = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 20],
    "clf__min_samples_leaf": [1, 2, 5, 10, 20, 30],
    "clf__max_features": ["sqrt", "log2"],
}

# Preprocessing sits inside the pipeline so it gets refit on each fold.
# Fitting it once on all of 1994 beforehand lets every fold's medians and
# category lists see the held-out third, which slightly flatters the search.
tune_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced",
                                   random_state=42, n_jobs=-1)),
])

grid_search = GridSearchCV(
    estimator=tune_pipe,
    param_grid=param_grid,
    scoring="average_precision",
    cv=3,
    verbose=1,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)   # raw, not preprocessed

print("Best parameters:", grid_search.best_params_)
print(f"Best CV PR-AUC: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__n_estimators': 500}
Best CV PR-AUC: 0.2357


### Evaluating the Optimized Random Forest

Now, let's evaluate this tuned model on our validation set and compare it to our previous results.

In [7]:
# No re-definition of evaluate here. Reusing the one defined earlier keeps the
# column names consistent, which is why the old version produced a table of NaN.
results.append(evaluate("RF, tuned", grid_search.best_estimator_,
                        X_train, y_train, X_val, y_val))

results_df = pd.DataFrame(results).set_index("Approach").round(4)

print("RANKING QUALITY, every model.\n")
display(results_df[["PR-AUC", "Caught top 10%", "Caught top 20%"]]
        .sort_values("PR-AUC", ascending=False))

print(f"\n\nAT A CUTOFF THAT FLAGS THE TOP {int(CAPACITY*100)}% OF CLAIMS.\n")
display(results_df[["Precision @cap", "Recall @cap", "F1 @cap"]]
        .sort_values("F1 @cap", ascending=False))

RANKING QUALITY, every model.



,PR-AUC,Caught top 10%,Caught top 20%
Approach,,,
"RF, tuned",0.1556,0.3223,0.5415
"RF, fully grown trees",0.1556,0.3223,0.5415
"LR, no handling",0.1488,0.2824,0.5216
"RF, balanced_subsample",0.1453,0.2857,0.5116
"RF, class_weight balanced",0.1448,0.2857,0.5183
"RF, no class weight",0.1431,0.2724,0.5216
"LR, class_weight balanced",0.1393,0.2724,0.5083




AT A CUTOFF THAT FLAGS THE TOP 20% OF CLAIMS.



,Precision @cap,Recall @cap,F1 @cap
Approach,,,
"RF, tuned",0.1536,0.5415,0.2394
"RF, fully grown trees",0.1536,0.5415,0.2394
"LR, no handling",0.1511,0.5216,0.2343
"RF, no class weight",0.1511,0.5216,0.2343
"RF, class_weight balanced",0.1501,0.5183,0.2328
"RF, balanced_subsample",0.1482,0.5116,0.2299
"LR, class_weight balanced",0.1473,0.5083,0.2284


Top 15 features by random forest importance (mean decrease in impurity):

In [8]:
# Importances come from the model we are actually shipping, the tuned
# fully grown forest, not the leaf-limited one used earlier.
best_rf = grid_search.best_estimator_.named_steps["clf"]
best_names = grid_search.best_estimator_.named_steps["pre"].get_feature_names_out()
imp = pd.Series(best_rf.feature_importances_, index=best_names)

print("Top 15 individual columns after one-hot encoding:\n")
display(imp.sort_values(ascending=False).head(15).round(4))

# Roll the one-hot columns back up to the original column they came from.
# This is the view that matters, because a column split into 8 one-hot
# columns looks unimportant even when the underlying thing is not.
cols_by_len = sorted(MODEL_FEATURES, key=len, reverse=True)


def source_col(name):
    body = name.split("__", 1)[1]
    for c in cols_by_len:
        if body == c or body.startswith(c + "_"):
            return c
    return body


grouped_imp = (imp.groupby(imp.index.map(source_col))
                  .sum()
                  .sort_values(ascending=False))

print("\n\nImportance rolled up to the original columns:\n")
display(grouped_imp.round(4))
print("\nShould sum to 1.0:", round(grouped_imp.sum(), 4))

Top 15 individual columns after one-hot encoding:



,0
numeric__Age,0.0695
categorical__Fault_Third Party,0.0613
categorical__Fault_Policy Holder,0.0549
categorical__BasePolicy_Liability,0.0534
numeric__WeekOfMonth,0.0382
numeric__WeekOfMonthClaimed,0.0364
numeric__DriverRating,0.0346
categorical__VehicleCategory_Sport,0.0339
categorical__BasePolicy_All Perils,0.0325
categorical__VehicleCategory_Sedan,0.0282




Importance rolled up to the original columns:



,0
Fault,0.1161
BasePolicy,0.1062
DayOfWeek,0.0763
Make,0.0741
Age,0.0695
VehicleCategory,0.0649
DayOfWeekClaimed,0.0635
PastNumberOfClaims,0.0507
AgeOfPolicyHolder,0.0491
AgeOfVehicle,0.0483



Should sum to 1.0: 1.0


In [9]:
# CA's point against the month columns was that the pattern reflects
# when the insurer ran investigations, not when fraud happened. The same
# argument applies to day of week and week of month, and the fully grown
# forest now leans on all four. Testing whether they earn their place.
CALENDAR = ["DayOfWeek", "DayOfWeekClaimed", "WeekOfMonth", "WeekOfMonthClaimed"]


def make_pre(feats):
    num = X_train[feats].select_dtypes(include="number").columns.tolist()
    cat = X_train[feats].select_dtypes(exclude="number").columns.tolist()
    grouped = [c for c in ["Make", "PolicyType"] if c in cat]
    regular = [c for c in cat if c not in grouped]
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())]), num),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))]), regular),
        ("grouped_categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=0.01))]), grouped),
    ])


calendar_sets = {
    "current 24": MODEL_FEATURES,
    "minus 4 calendar columns": [c for c in MODEL_FEATURES if c not in CALENDAR],
}

calendar_rows = []
for name, feats in calendar_sets.items():
    pipe = Pipeline([
        ("pre", make_pre(feats)),
        ("clf", RandomForestClassifier(n_estimators=500,
                                       class_weight="balanced",
                                       random_state=42, n_jobs=-1)),
    ])
    row = evaluate(name, pipe, X_train[feats], y_train, X_val[feats], y_val)
    row["features"] = len(feats)
    calendar_rows.append(row)

cal_df = pd.DataFrame(calendar_rows).set_index("Approach").round(4)
display(cal_df[["features", "PR-AUC", "Caught top 10%", "Caught top 20%"]])

,features,PR-AUC,Caught top 10%,Caught top 20%
Approach,,,,
current 24,24,0.1556,0.3223,0.5415
minus 4 calendar columns,20,0.1423,0.2857,0.5017


## Task 3 summary: random forest

Trained on 1994, scored on 1995, using the 24 column feature list from Task 1.
The 1996 test year was not touched. Both logistic regression rows are rerun
here on the same features so the comparison is like for like.

### Ranking quality on 1995

| Model | PR-AUC | Caught top 10% | Caught top 20% |
|---|---|---|---|
| **RF, tuned (= fully grown)** | **0.1556** | **0.3223** | **0.5415** |
| LR, no handling | 0.1488 | 0.2824 | 0.5216 |
| RF, balanced_subsample | 0.1453 | 0.2857 | 0.5116 |
| RF, class_weight balanced | 0.1448 | 0.2857 | 0.5183 |
| RF, no class weight | 0.1431 | 0.2724 | 0.5216 |
| LR, class_weight balanced | 0.1393 | 0.2724 | 0.5083 |

The random forest is the best model, catching 54.2% of 1995's fraud in the top
20% of scored claims against logistic regression's 52.2%.

### The leaf size limit was costing us

`min_samples_leaf=20` was originally added because fully grown trees rarely
score a claim above 0.5, which makes recall at that threshold collapse to
0.0033. That diagnosis was right, but it is a problem with using 0.5 rather
than a problem with the model. Judged on how it orders claims, the fully grown
forest is the best model here, and limiting leaf size cost 0.0108 PR-AUC and
2.3 points of top 20% capture to fix a symptom of a threshold we do not use.

### Tuning found the same model

Grid search over 48 combinations, optimising average precision with 3-fold
cross-validation inside 1994, selected `min_samples_leaf=1`, `max_depth=None`,
`max_features='sqrt'`, `n_estimators=500`. That is the fully grown forest, so
the tuned and untuned rows are identical. The original grid searched
`min_samples_leaf` from 10 upward and could not reach it.

Note on the reported CV score of 0.2357: that is not comparable to the 0.1556
on 1995. Cross-validation inside 1994 trains and tests on the same year, so it
never has to cope with the year-to-year drift that a real deployment does. The
1995 number is the honest one.

### Class weighting makes no real difference to the forest either

At leaf size 20, the forest scores 0.1448 with `balanced`, 0.1431 with no
weighting, and 0.1453 with `balanced_subsample`. All within 0.002, which is
noise on 301 fraud cases. This matches Task 2's finding on logistic
regression, and it is now tested on trees rather than assumed. The final model
keeps `class_weight="balanced"` since it costs nothing.

### What the model uses

Rolled up from the one-hot columns to the original columns, Fault at 0.2331,
BasePolicy at 0.2193 and VehicleCategory at 0.1327 carry 58% of the model
between them. No month column appears anywhere, and with PolicyType dropped
the credit for coverage type is no longer split three ways, which is what the
Challenge Advisor asked for ahead of SHAP in November.

Two columns come out at exactly 0.0000, Days_Policy_Accident and
Days_Policy_Claim, so the forest never splits on them at all. ClaimLagMonths
comes in at 0.0159, so the forest does use it even though it made no
difference to logistic regression.

### Caveats

- The forest beats logistic regression by 0.0068 PR-AUC on 301 fraud cases.
  That is consistent across all three ranking metrics, which helps, but it is
  not a wide enough gap to call a clear winner without confidence intervals.
- Precision at a 20% capacity cutoff is 0.1536, so roughly one in six flagged
  claims is fraud. Better than the 1 in 19 base rate, but still a lot of clean
  claims being opened.
- Hyperparameters were chosen with 3-fold cross-validation inside 1994, which
  shuffles within the year. A time-aware split would be stricter.

### We tested extending the month argument to other calendar columns

The Challenge Advisor's case against the month columns was that the pattern
reflects when the insurer ran investigations rather than when fraud happened,
so it will not repeat. The fully grown forest leans fairly heavily on
DayOfWeek, DayOfWeekClaimed, WeekOfMonth and WeekOfMonthClaimed, so we tested
whether the same argument applies to them.

| Feature set | Features | PR-AUC | Caught top 10% | Caught top 20% |
|---|---|---|---|---|
| Current list | 24 | **0.1556** | **0.3223** | **0.5415** |
| Minus the 4 calendar columns | 20 | 0.1423 | 0.2857 | 0.5017 |

It does not apply. Removing them costs 0.0133 PR-AUC and 4 points of top 20%
capture, so they are kept.

The difference is how often each cycle repeats inside the training year. 1994
contains exactly one January, so a monthly pattern rests on a single month of
data with nothing to average out an investigation push. Day of week repeats
about 52 times in the same year, so a weekly pattern has roughly 50 times more
evidence behind it, and the test shows it carries into 1995. If it were
1994-specific noise it would have hurt rather than helped.

A weekly rhythm is also plausible on its own terms, through weekend accidents
and Monday morning filings, in a way that "October" is not.

### A caveat on the importances themselves

These are mean decrease in impurity, which is biased toward columns offering
many possible split points. A continuous column like Age can be split at
hundreds of values while AgentType can only be split one way, so Age
accumulates importance partly for that reason. A fully grown forest makes the
bias stronger. Permutation importance does not have this problem and should be
used for the November interpretability work alongside SHAP.